In [ ]:
import polars as pl
import pandas as pd
import xlsxwriter
from pathlib import Path
from typing import List, Optional

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)

# ==============================================================================
# CONFIGURATION
# ==============================================================================
BASE_PATH = Path(r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\Text_Mining")
PARQUET_PATH = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\OUTPUT_PERFORMANCE\OUTPUT_PERFORMANCE_COMBINE\_performance_hcm.parquet"
SRC_DIR   = BASE_PATH / '0_source'
OUT_DIR   = BASE_PATH / '1_des'
PER_DIR   = PARQUET_PATH

# --- USER SELECTION: Define specific files to process here ---
TARGET_CSV_FILES = ["Car_Transcript_CNX 2026-06_01_08.csv"] 

PARQUET_FILE = SRC_DIR / "Combined_Transcripts_Optimized.parquet"
PER_FILE = pl.read_parquet(PER_DIR).select(['Conversation Id', 'Agent Email ID', '_promoter', '_detractor', '_neutral', '_survey', '_lc', 'Handle Time (Sum)', '_verbatim'])

# Ensure directories exist
SRC_DIR.mkdir(exist_ok=True, parents=True)
OUT_DIR.mkdir(exist_ok=True, parents=True)

# Keyword Mapping Strategy
KEYWORDS_MAPPING = {

    # "Age issue": r"minimum age|\b18\b|\b20\b|\b21\b|\b25\b|age surcharge|young driver|underage|underage fee|under 25|over 70|age thresholds|age requirement|too young to rent|years old|meet age requirement",
    # "Driver's License Issue": r"driver lisence|driver.?s license|\blicense\b|\blicence\b|license issue|license requirement|\bidp\b|international license|international driver.?s license|\bdl\b|temporary license|national license|eu license|wrong name|expired license|driving licence|driving license|valid license|invalid license|my license is not valid|\bdmv\b|home country driving licence|international driving permit",
    # "Credit/Debit Card Issue": r"credit card|debit card|chime card|card issue|don.?t accept debit card|didn.?t accept debit card|declined my card|card requirement|require a credit card|don.?t have a credit card|ask for a credit card|want a credit card|no credit card|card not valid|credit check|booked using my debit card|physical card|major card|\bcard\b"
    # # "Mileage": r"unlimited mileage|unlimited miles|unl miles|unl mileage|unlimited mile|unl mile|mileage",
    # # "Underage": r"young driver|minimum age|underage surcharge|underage|age limit|age restriction|driver.?s age|age policy|age requirement|age surcharge|young renter|underage driver|age exception|age eligibility|rental age rule|special age condition|age limit policy|legal driving age|too young to rent|age cutoff|driver age rule|young renter restriction|age disqualification|rental age requirement|age compliance|driver age criteria|age threshold",
    # "Deposit issue": r"available funds|insufficient funds|temporary hold|deposit|potential charge|high deposit|large deposit|unexpected deposit|hidden deposit|extra deposit|deposit too high|deposit amount|deposit requirement|deposit policy|deposit hold|no funds|authorization hold|deposit not possible|unable to pay deposit|ridiculous deposit|unfair deposit|deposit not mentioned|card declined for deposit|deposit obligation|deposit scam",
    # "Credit card issue": r"credit card required|no credit card|only have debit card|debit card not accepted|card requirement|credit card policy|credit card rule|need credit card|without credit card|credit card mandatory|why credit card|not informed about credit card|unexpected credit card demand|debit card refused|card not accepted|card problem at counter|card requirement not informed|only credit card allowed|card restriction|card eligibility|card condition|card refused at counter",
    # "Mexican": r"mexican liability|mexico liability"
    # "Change Time": r"pick.?up time|collection time|start time|drop.?off time|return time|end time|flight delay|delayed flight|flight late|missed flight|wrong time|incorrect time|mistake time|change time|modify time|reschedule|adjust time|wrong driver|incorrect driver|wrong name|incorrect name|spelling error|typo|name mistake",
    # "Change Car Type": r"upgrade|upsell|larger car|bigger car|more space|suv|downgrade|smaller car|compact|economy|electric|ev car|tesla|automatic|transmission|auto car|hybrid|change car|switch car|swap car|swap vehicle|different car|different vehicle|wrong car type"
    #"Risk free booking ":r"risk free booking|risk free",
    #"Insurance Issues": r"did not honor insurance|insurance not honored|refund my insurance|vendor not accept expedia insurance|vendor request buy insurance from them|expedia insurance not valid|your insurance insufficient|expedia insurance does not cover anything",

    "Vendor not accept Expedia insurance": (
    r"vendor.{0,15}not.{0,10}accept.{0,15}insurance"        # vendor not accept ... insurance
    r"|vendor.{0,15}not.{0,10}honor.{0,15}insurance"         # vendor not honor ... insurance
    r"|vendor.{0,15}reject.{0,15}insurance"                  # vendor reject insurance
    r"|vendor.{0,15}refused.{0,15}insurance"                 # vendor refused insurance
    r"|vendor.{0,15}don.?t.{0,10}accept.{0,15}insurance"     # vendor don't accept insurance
    r"|vendor.{0,15}didn.?t.{0,10}accept.{0,15}insurance"    # vendor didn't accept insurance
    r"|vendor.{0,15}won.?t.{0,10}accept.{0,15}insurance"     # vendor won't accept insurance
    r"|car.{0,10}rental.{0,15}not.{0,10}accept.{0,15}insurance"
    r"|rental.{0,15}not.{0,10}accept.{0,15}insurance"
    r"|insurance.{0,15}not.{0,10}accepted"                   # insurance not accepted
    r"|insurance.{0,15}not.{0,10}honored"                    # insurance not honored
    r"|insurance.{0,15}not.{0,10}valid"                      # insurance not valid
    r"|insurance.{0,15}invalid"                              # insurance invalid
    r"|invalid.{0,10}insurance"                              # invalid insurance
    r"|insurance.{0,15}not.{0,10}recognized"                 # insurance not recognized
    r"|did.{0,10}not.{0,10}honor.{0,15}insurance"            # did not honor insurance
    r"|not.{0,10}honoring.{0,15}insurance"                   # not honoring insurance
    r"|expedia.{0,15}insurance.{0,15}not"                    # expedia insurance not ...
    r"|expedia.{0,15}insurance.{0,15}different"              # expedia insurance different
    r"|expedia.{0,15}insurance.{0,15}invalid"                # expedia insurance invalid
    r"|expedia.{0,15}insurance.{0,15}insuffi"                # expedia insurance insufficient
    r"|expedia.{0,15}insurance.{0,15}doesn.?t.{0,10}cover"   # expedia insurance doesn't cover
    r"|expedia.{0,15}insurance.{0,15}not.{0,10}cover"        # expedia insurance not cover
    r"|expedia.{0,15}insurance.{0,15}useless"                # expedia insurance useless
    r"|\beg\b.{0,15}insurance.{0,15}different"               # EG insurance different
    r"|\beg\b.{0,15}insurance.{0,15}not"                     # EG insurance not ...
    r"|insurance.{0,10}from.{0,10}expedia.{0,15}not"         # insurance from expedia not ...
    r"|buy.{0,15}insurance.{0,15}from.{0,15}them"            # buy insurance from them (vendor)
    r"|purchase.{0,15}insurance.{0,15}from.{0,15}them"       # purchase insurance from them
    r"|get.{0,15}insurance.{0,15}from.{0,15}counter"         # get insurance from counter
    r"|insurance.{0,15}at.{0,15}counter"                     # insurance at counter
    r"|counter.{0,15}insurance"                              # counter insurance
    r"|own.{0,15}insurance.{0,15}policy"                     # own insurance policy
    r"|basic.{0,15}insurance.{0,15}not"                      # basic insurance not ...
    r"|not.{0,10}see.{0,15}basic.{0,15}insurance"            # not see basic insurance
    r"|won.?t.{0,10}able.{0,10}to.{0,10}see.{0,15}insurance" # won't able to see insurance
    r"|can.?t.{0,10}see.{0,15}insurance"                     # can't see insurance
    r"|cannot.{0,10}see.{0,15}insurance"                     # cannot see insurance
    r"|insurance.{0,15}not.{0,10}show"                       # insurance not showing
    r"|refund.{0,15}insurance"                               # refund insurance
    r"|insurance.{0,15}refund"                               # insurance refund
),
}

In [ ]:
def process_final_pipeline_clean(
    src_dir: Path, 
    out_dir: Path, 
    parquet_path: Path, 
    keyword_map: dict, 
    per_df: pl.DataFrame,
    target_files: List[str]
):
    # ==========================================================================
    # STEP 1: ETL (CSV -> PARQUET) WITH FILE SELECTION
    # ==========================================================================
    print("--- STEP 1: CONVERTING CSV TO PARQUET ---")
    
    files_to_read = []
    
    if target_files:
        print(f"-> Mode: Selected Files Only ({len(target_files)} files)")
        for fname in target_files:
            file_path = src_dir / fname
            if file_path.exists():
                files_to_read.append(str(file_path))
            else:
                print(f"[!] Warning: File not found: {fname}")
    else:
        print("-> Mode: All CSV Files in Directory")
        files_to_read = [str(p) for p in src_dir.glob("*.csv")]

    if not files_to_read and not parquet_path.exists():
        print("[!] No source files found to process. Exiting.")
        return

    if files_to_read:
        print(f"-> Found {len(files_to_read)} valid files. Merging...")
        try:
            q = pl.scan_csv(files_to_read, ignore_errors=True)
            df_combined = q.collect()
            df_combined.write_parquet(parquet_path, compression="snappy")
            print(f"-> Saved optimized file: {parquet_path.name}")
        except Exception as e:
            print(f"[!] Error merging CSVs: {e}")
            return
    else:
        print("-> Using existing Parquet file (No new CSVs processed).")

    # ==========================================================================
    # STEP 2: LOADING & PRE-PROCESSING
    # ==========================================================================
    print("\n--- STEP 2: LOADING & PRE-PROCESSING ---")
    df_raw = pl.read_parquet(parquet_path)
    
    df_base = df_raw.filter(pl.col("Participant Type") == "HumanAgent")

    if df_base.is_empty():
        print("[!] No 'HumanAgent' data found.")
        return

    df_prep = df_base.with_columns([
        pl.col("Text").str.to_lowercase().alias("text_lower"),
        pl.col("Joined Time").str.to_datetime(strict=False).dt.date().alias("Date")
    ])

    # ==========================================================================
    # STEP 3: MINING (EXPLODE KEYWORDS)
    # ==========================================================================
    print("\n--- STEP 3: MINING & EXPLODING KEYWORDS ---")
    
    dfs_to_concat = []
    
    group_keys = [
        "Conversation Id", "Joined Time", "Date",
        "Agent People Id", "Agent Email ID",
        "Latest VA Product", "Latest VA Intent", "Agent Vendor Location", "Pickup Date Time"
    ]

    for article_name, pattern in keyword_map.items():
        df_hits = df_prep.filter(pl.col("text_lower").str.contains(pattern))
        
        if df_hits.is_empty():
            continue

        df_agg = (
            df_hits
            .with_columns(
                pl.col("text_lower").str.extract_all(pattern).alias("extracted_keywords_list")
            )
            .explode("extracted_keywords_list")
            .group_by(group_keys + ["extracted_keywords_list"])
            .agg([
                pl.len().alias("Total"),
                pl.col("Sent Time").min().alias("Sent Time"),
                pl.col("Text").unique().str.join(" | ").alias("Matched_Content")
            ])
            .with_columns([
                pl.lit(article_name).alias("Articles"),
                pl.lit(1).cast(pl.Int8).alias("Found"),
                pl.col("extracted_keywords_list").alias("Keywords")
            ])
            .drop("extracted_keywords_list")
        )
        dfs_to_concat.append(df_agg)

    # ==========================================================================
    # STEP 4: MERGE & EXPORT
    # ==========================================================================
    if not dfs_to_concat:
        print("[!] No keywords found.")
        return

    df_final = pl.concat(dfs_to_concat)
    
    print(f"-> Merging with Performance Data (Shape before: {df_final.shape})")
    
    df_final = df_final.with_columns([
        pl.col("Conversation Id").cast(pl.Utf8),
        pl.col("Agent Email ID").cast(pl.Utf8)
    ])
    
    per_df_clean = per_df.with_columns([
        pl.col("Conversation Id").cast(pl.Utf8),
        pl.col("Agent Email ID").cast(pl.Utf8)
    ])

    df_final = df_final.join(
        per_df_clean,
        on=['Conversation Id', 'Agent Email ID'],
        how='left'
    )
    print(f"-> Merge complete (Shape after: {df_final.shape})")

    perf_cols = ['_promoter', '_detractor', '_neutral', '_survey', '_lc', 'Handle Time (Sum)', '_verbatim']
    priority_cols = group_keys + ["Articles", "Keywords", "Found", "Total", "Sent Time"] + perf_cols
    
    df_final = df_final.select(
        [c for c in priority_cols if c in df_final.columns] + 
        [c for c in df_final.columns if c not in priority_cols]
    ).sort("Date")

    # A) Export Parquet
    out_parquet = out_dir / "text_mining_keywords_exploded.parquet"
    df_final.write_parquet(out_parquet, compression="snappy")
    print(f"-> Exported Parquet: {out_parquet.name}")

    # B) Export Excel — one file per Article
    print(f"-> Converting to Excel (grouped by Articles)...")

    df_pandas = df_final.to_pandas()

    if "Matched_Content" in df_pandas.columns:
        df_pandas["Matched_Content"] = df_pandas["Matched_Content"].astype(str).str.slice(0, 30000)

    import numpy as np
    df_pandas = df_pandas.replace([np.inf, -np.inf], np.nan)

    for article_name, df_group in df_pandas.groupby("Articles"):

        safe_name = (
            str(article_name)
            .replace("/", "-").replace("\\", "-").replace(":", "-")
            .replace("*", "").replace("?", "").replace('"', "")
            .replace("<", "").replace(">", "").replace("|", "")
        )
        out_excel = out_dir / f"text_mining_{safe_name}.xlsx"

        try:
            with pd.ExcelWriter(out_excel, engine='xlsxwriter') as writer:
                writer.book.strings_to_urls = False
                df_group.to_excel(writer, index=False, sheet_name=safe_name[:31])
            print(f"-> Exported Excel: {out_excel.name}  ({len(df_group)} rows)")

        except Exception as e:
            print(f"[!] Error saving Excel for '{article_name}': {e}")

    print(f"-> Total Rows: {len(df_final)}")
    print(f"-> SUCCESS.")


# Execute
if __name__ == "__main__":
    process_final_pipeline_clean(SRC_DIR, OUT_DIR, PARQUET_FILE, KEYWORDS_MAPPING, PER_FILE, TARGET_CSV_FILES)

--- STEP 1: CONVERTING CSV TO PARQUET ---
-> Mode: Selected Files Only (1 files)
-> Found 1 valid files. Merging...
-> Saved optimized file: Combined_Transcripts_Optimized.parquet

--- STEP 2: LOADING & PRE-PROCESSING ---

--- STEP 3: MINING & EXPLODING KEYWORDS ---
-> Merging with Performance Data (Shape before: (56, 15))
-> Merge complete (Shape after: (64, 22))
-> Exported Parquet: text_mining_keywords_exploded.parquet
-> Converting to Excel (grouped by Articles)...
-> Exported Excel: text_mining_Vendor not accept Expedia insurance.xlsx  (64 rows)
-> Total Rows: 64
-> SUCCESS.


In [15]:
# import sys

# CSV_FILE_PATH = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\Text_Mining\0_source\Car.csv"

# def copy_vendor_counts_to_clipboard(file_path: str):
#     try:
#         q = (
#             pl.scan_csv(file_path, ignore_errors=True)
#             .filter(
#                 pl.col("Agent Vendor Location").is_not_null() & 
#                 pl.col("Agent Email ID").is_not_null() &
#                 pl.col("Conversation Id").is_not_null()
#             )
#             .select(["Agent Vendor Location", "Conversation Id", "Agent Email ID"])
#             .unique()
#             .group_by("Agent Vendor Location")
#             .agg([
#                 pl.len().alias("Vol")
#             ])
#             .sort("Vol", descending=True)
#         )
        
#         df_result = q.collect()

#         total_vol = df_result["Vol"].sum()

#         total_row = pl.DataFrame(
#             {
#                 "Agent Vendor Location": ["Grand Total"], 
#                 "Vol": [total_vol]
#             },
#             schema=df_result.schema 
#         )

#         df_final = pl.concat([df_result, total_row])
        
#         try:
#             df_final.write_clipboard(separator="\t")
#         except Exception:
#             pass
        
#         return df_final

#     except Exception:
#         return None

# if __name__ == "__main__":
#     if Path(CSV_FILE_PATH).exists():
#         copy_vendor_counts_to_clipboard(CSV_FILE_PATH)

In [16]:
# TRACKING_FILE_XLSX = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\Text_Mining\C&R Tracking Internal & External.xlsx"
# PER_DIR = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\RAW\OUTPUT_PERFORMANCE\OUTPUT_PERFORMANCE_COMBINE\_performance_hcm.parquet"

# if not Path(TRACKING_FILE_XLSX).exists():
#     raise FileNotFoundError(f"File not found: {TRACKING_FILE_XLSX}")

# df_tracking = pl.read_excel(
#     TRACKING_FILE_XLSX, 
#     sheet_name="Internal Tracking",
#     engine="openpyxl",
#     infer_schema_length=0 
# )

# df_tracking = df_tracking.with_columns(
#     pl.col("Conversation ID").cast(pl.Utf8).str.strip_chars()
# )

# df_perf = pl.read_parquet(PER_DIR)

# cols_to_add = [
#     'Conversation Id', 
#     'Agent Email ID', 
#     '_promoter', '_detractor', '_neutral', 
#     '_survey', '_lc', 'Handle Time (Sum)', 'NPS'
# ]

# df_perf_clean = (
#     df_perf
#     .select([c for c in cols_to_add if c in df_perf.columns])
#     .with_columns(
#         pl.col("Conversation Id").cast(pl.Utf8).str.strip_chars()
#     )
#     .unique(subset=["Conversation Id"], keep="first")
# )

# df_final = df_tracking.join(
#     df_perf_clean,
#     left_on=["Email","Conversation ID"],
#     right_on=["Agent Email ID","Conversation Id"],
#     how="left",
#     coalesce=True 
# )

# output_path = Path(TRACKING_FILE_XLSX).parent / "Internal_Tracking_Performance_Result.xlsx"
# df_final.write_excel(output_path)